In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('XYZ_analysis.csv')

In [5]:
cols = df.columns.tolist()
start_idx = cols.index('ABC_Class') + 1
end_idx = cols.index('Mean_Demand') if 'Mean_Demand' in cols else len(cols)
demand_columns = cols[start_idx:end_idx]

print(f"Detected {len(demand_columns)} demand columns: {demand_columns}")

Detected 12 demand columns: ['Jan-25', 'Feb-25', 'Mar-25', 'Apr-25', 'May-25', 'Jun-25', 'Jul-25', 'Aug-25', 'Sep-25', 'Oct-25', 'Nov-25', 'Dec-25']


In [6]:
# ====================== 2. CALCULATE MEAN, STD DEV & CV ======================
df['Mean_Demand'] = df[demand_columns].mean(axis=1)
df['Std_Dev']     = df[demand_columns].std(axis=1, ddof=1)   # sample std dev (same as Excel)
df['CV']          = df['Std_Dev'] / df['Mean_Demand']
df['CV']          = df['CV'].fillna(0)                       # avoid division by zero

In [7]:
# ====================== 3. ASSIGN XYZ CLASS ======================
def get_xyz(cv):
    if cv <= 0.25:
        return 'X'      # Stable
    elif cv <= 0.5:
        return 'Y'      # Moderate
    else:
        return 'Z'      # Erratic

df['XYZ_Class'] = df['CV'].apply(get_xyz)

In [8]:
# ====================== 4. CREATE COMBINED ABC+XYZ ======================
df['ABC_XYZ_Combined'] = df['ABC_Class'] + df['XYZ_Class']

# ====================== 5. SHOW RESULTS ======================
print("\n=== DETAILED XYZ RESULTS ===")
print(df[['SKU', 'ABC_Class', 'Mean_Demand', 'Std_Dev', 'CV', 
          'XYZ_Class', 'ABC_XYZ_Combined']].round(4))


=== DETAILED XYZ RESULTS ===
      SKU ABC_Class  Mean_Demand  Std_Dev      CV XYZ_Class ABC_XYZ_Combined
0  SKU_A1         A     119.8333   1.7495  0.0146         X               AX
1  SKU_A2         A     140.8333  92.2160  0.6548         Z               AZ
2  SKU_B1         B      50.2500   1.2154  0.0242         X               BX
3  SKU_B2         B      50.5833  18.8412  0.3725         Y               BY
4  SKU_C1         C      10.2500   1.2154  0.1186         X               CX
5  SKU_C2         C      15.2500  12.5996  0.8262         Z               CZ


In [9]:
# ====================== 6. BUILD THE 9-CELL ABC-XYZ MATRIX ======================
print("\n=== ABC-XYZ MATRIX (Counts) ===")
matrix = pd.crosstab(df['ABC_Class'], df['XYZ_Class'], margins=True, margins_name='Total')
print(matrix)

print("\n=== ABC-XYZ MATRIX (% of total SKUs) ===")
percent_matrix = (matrix / matrix.loc['Total', 'Total'] * 100).round(1)
print(percent_matrix)


=== ABC-XYZ MATRIX (Counts) ===
XYZ_Class  X  Y  Z  Total
ABC_Class                
A          1  0  1      2
B          1  1  0      2
C          1  0  1      2
Total      3  1  2      6

=== ABC-XYZ MATRIX (% of total SKUs) ===
XYZ_Class     X     Y     Z  Total
ABC_Class                         
A          16.7   0.0  16.7   33.3
B          16.7  16.7   0.0   33.3
C          16.7   0.0  16.7   33.3
Total      50.0  16.7  33.3  100.0


In [10]:
# ====================== 7. SAVE RESULTS ======================
with pd.ExcelWriter('XYZ_analysis_python_results.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='XYZ_Data', index=False)
    matrix.to_excel(writer, sheet_name='ABC_XYZ_Matrix')
    percent_matrix.to_excel(writer, sheet_name='ABC_XYZ_Percent')

print("\n✅ All results saved to 'XYZ_analysis_python_results.xlsx' 🎉")
print("Open this new Excel file to see the full report!")


✅ All results saved to 'XYZ_analysis_python_results.xlsx' 🎉
Open this new Excel file to see the full report!
